# Model 4: k-nearest neighbours

This notebook builds and tunes kNN models.

In [1]:
from pathlib import Path
import json
import sys

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import get_split
from src.evaluate import cv_report, save_result
from src.pipeline import build

REPORTS_DIR = PROJECT_ROOT / 'reports'
BEST_PARAMS_PATH = REPORTS_DIR / 'best_params.json'
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Split first

In [2]:
X_train, X_test, y_train, y_test = get_split()
print(f'Training rows: {len(X_train):,}')
print(f'Held-out test rows: {len(X_test):,}')
print(f'Training yes-rate: {y_train.mean():.3%}')

Training rows: 36,168
Held-out test rows: 9,043
Training yes-rate: 11.698%


## k-nearest neighbours baseline and tuning

In [5]:
# The shared pipeline scales numeric columns before kNN calculates distances.
knn_baseline = build(KNeighborsClassifier(n_neighbors=15, n_jobs=-1))
knn_baseline_row = cv_report('KNeighbors', knn_baseline, X_train, y_train)
save_result(knn_baseline_row, REPORTS_DIR / 'model_comparison.csv')
knn_baseline_row

{'name': 'KNeighbors',
 'pr_auc_mean': np.float64(0.33551636758450865),
 'pr_auc_std': np.float64(0.011747105643679326),
 'roc_auc_mean': np.float64(0.7291438973237098),
 'roc_auc_std': np.float64(0.005469503370412958),
 'f1_mean': np.float64(0.2564344507180429),
 'f1_std': np.float64(0.011336178847175965),
 'precision_mean': np.float64(0.6075185327033998),
 'precision_std': np.float64(0.029026018690497674),
 'recall_mean': np.float64(0.1626092368838984),
 'recall_std': np.float64(0.007966076905706814),
 'train_pr_auc_mean': np.float64(0.4512022420285395),
 'train_pr_auc_std': np.float64(0.002986654661786897),
 'fit_time_mean': np.float64(0.1353611946105957)}

In [6]:
knn_search = GridSearchCV(
    estimator=build(KNeighborsClassifier(n_jobs=-1)),
    param_grid={
        'model__n_neighbors': [5, 15, 31, 51],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2],
    },
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)
knn_tuned_row = cv_report('KNeighbors-tuned', knn_search.best_estimator_, X_train, y_train)
save_result(knn_tuned_row, REPORTS_DIR / 'model_comparison.csv')
print(knn_search.best_params_)
knn_tuned_row

{'model__n_neighbors': 51, 'model__p': 1, 'model__weights': 'distance'}


{'name': 'KNeighbors-tuned',
 'pr_auc_mean': np.float64(0.3887053241589263),
 'pr_auc_std': np.float64(0.010675130269362683),
 'roc_auc_mean': np.float64(0.754646074783374),
 'roc_auc_std': np.float64(0.0012793498224640777),
 'f1_mean': np.float64(0.23764979939351485),
 'f1_std': np.float64(0.011928189933326777),
 'precision_mean': np.float64(0.671732751188115),
 'precision_std': np.float64(0.030047119071883486),
 'recall_mean': np.float64(0.14440899740706317),
 'recall_std': np.float64(0.007865247739164657),
 'train_pr_auc_mean': np.float64(1.0),
 'train_pr_auc_std': np.float64(0.0),
 'fit_time_mean': np.float64(0.07482891082763672)}

## Preserve the best-parameter record

In [ ]:

if BEST_PARAMS_PATH.exists():
    best_params = json.loads(BEST_PARAMS_PATH.read_text())
else:
    best_params = {}
best_params['KNeighbors-tuned'] = knn_search.best_params_
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2) + '\n')
print(json.dumps(best_params, indent=2))
